<a href="https://colab.research.google.com/github/rajesh0305/Generative-and-Agentic-AI-Project/blob/main/student_lab_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6: Reasoning Agents: Inference-Time Compute and Search

Earlier labs focused on how we *train* models to behave differently. In this lab, we study a different question: **how can we make the same model reason better at inference time by changing the control loop around it?**

You will implement and compare four approaches to solving the **Game of 24**, a classic arithmetic reasoning task:

1. **Single-shot prompting**: one prompt, one answer
2. **Chain-of-thought prompting**: elicit step-by-step reasoning
3. **Self-consistency / best-of-N**: sample multiple reasoning traces, majority vote
4. **Tree-of-Thought reasoning agent**: a control loop that proposes, evaluates, and searches over candidate solutions

The key insight: **reasoning improvements can come from inference-time compute and search, not only from better training.**

---

## The Game of 24

**Rules:** Given four numbers (typically 1 to 13), use each number exactly once with basic arithmetic operations (+, -, \*, /) to make 24.

**Example:** Given `[1, 2, 3, 4]`, one solution is `1 * 2 * 3 * 4 = 24`.

This is an ideal testbed for reasoning agents because:
- Solutions are **objectively verifiable**: does the expression equal 24?
- The search space is **small enough** to explore but **large enough** that brute-force enumeration is tedious
- **Search genuinely helps**: harder puzzles benefit from exploring multiple approaches
- It mirrors the structure of real reasoning: propose a step, check if it's on track, backtrack if not

---

## Section 0: Setup with OpenRouter

We will use [OpenRouter](https://openrouter.ai) to access language models. OpenRouter provides a unified API that is compatible with the OpenAI SDK, so you can use the familiar `openai` Python package.

### 0.1 Create an OpenRouter Account

1. Go to [https://openrouter.ai](https://openrouter.ai) and sign up (Google/GitHub login works).
2. Navigate to [https://openrouter.ai/keys](https://openrouter.ai/keys) and click **Create Key**.
3. Copy your API key. It starts with `sk-or-v1-`.

### 0.2 Set Your API Key

Set your API key as an environment variable **before** launching the notebook:

```bash
export OPENROUTER_API_KEY="sk-or-v1-your-key-here"
```

Alternatively, you can paste it directly in the cell below (not recommended for shared environments).

In [ ]:
# =============================================================================
# Install dependencies (run once)
# =============================================================================

!pip install openai

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

import os
import re
import random
import itertools
from collections import Counter
from openai import OpenAI

SEED = 42
random.seed(SEED)

# ---------------------------------------------------------------------------
# OpenRouter setup
# ---------------------------------------------------------------------------
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    raise ValueError(
        "Please set your OPENROUTER_API_KEY environment variable.\n"
        "  export OPENROUTER_API_KEY='sk-or-v1-...'\n"
        "Or set it directly: OPENROUTER_API_KEY = 'sk-or-v1-...'"
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Model to use: a capable but affordable model
MODEL = "google/gemini-2.0-flash-001"

print(f"OpenRouter client ready.")
print(f"Model: {MODEL}")

In [ ]:
# =============================================================================
# PROVIDED: Helper function to call the model
# =============================================================================

def call_model(prompt, system_prompt=None, temperature=0.0, max_tokens=512):
    """Call a language model via OpenRouter.

    Args:
        prompt: The user message.
        system_prompt: Optional system message.
        temperature: Sampling temperature (0 = deterministic).
        max_tokens: Maximum tokens to generate.

    Returns:
        dict with keys:
            'content': the model's response text
            'prompt_tokens': number of input tokens
            'completion_tokens': number of output tokens
            'total_tokens': total tokens used
    """
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    usage = response.usage
    return {
        "content": response.choices[0].message.content,
        "prompt_tokens": usage.prompt_tokens if usage else 0,
        "completion_tokens": usage.completion_tokens if usage else 0,
        "total_tokens": usage.total_tokens if usage else 0,
    }


print("call_model() helper defined.")

In [ ]:
# =============================================================================
# Test your API connection
# =============================================================================

test_response = call_model("What is 2 + 2? Reply with just the number.")
print(f"Response: {test_response['content']}")
print(f"Tokens used: {test_response['total_tokens']}")
print("\nAPI connection working!")

---

## Section 0.5: Game of 24: Puzzles and Verification

Before we start prompting, we need two things:
1. A set of puzzles with known solutions (so we can measure accuracy)
2. A function that checks whether a proposed solution is valid

In [ ]:
# =============================================================================
# PROVIDED: Puzzle set
# =============================================================================

# Each puzzle has four numbers and a known valid solution.
# Difficulty: 'easy' has obvious solutions, 'hard' requires creative steps.
PUZZLES = [
    # ----- Easy -----
    {"numbers": [1, 2, 3, 4], "solution": "1 * 2 * 3 * 4",           "difficulty": "easy"},
    {"numbers": [6, 6, 6, 6], "solution": "6 + 6 + 6 + 6",           "difficulty": "easy"},
    {"numbers": [2, 3, 4, 5], "solution": "2 * (3 + 4 + 5)",         "difficulty": "easy"},

    # ----- Medium -----
    {"numbers": [4, 4, 4, 4], "solution": "4 * 4 + 4 + 4",           "difficulty": "medium"},
    {"numbers": [1, 3, 5, 7], "solution": "(7 + 5) * (3 - 1)",       "difficulty": "medium"},
    {"numbers": [2, 7, 8, 1], "solution": "2 * 8 + 7 + 1",           "difficulty": "medium"},
    {"numbers": [3, 7, 3, 7], "solution": "7 * (3 + 3 / 7)",         "difficulty": "medium"},

    # ----- Hard -----
    {"numbers": [1, 5, 5, 5], "solution": "5 * (5 - 1 / 5)",         "difficulty": "hard"},
    {"numbers": [3, 3, 8, 8], "solution": "8 / (3 - 8 / 3)",         "difficulty": "hard"},
    {"numbers": [1, 3, 4, 6], "solution": "6 / (1 - 3 / 4)",         "difficulty": "hard"},
]

print(f"Loaded {len(PUZZLES)} puzzles:")
for i, p in enumerate(PUZZLES):
    print(f"  {i+1:2d}. {str(p['numbers']):20s}  [{p['difficulty']}]")

In [ ]:
# =============================================================================
# PROVIDED: Solution verifier
# =============================================================================

def verify_solution(expression, numbers):
    """Check if an expression is a valid Game of 24 solution.

    A valid solution must:
    1. Use each of the given numbers exactly once
    2. Use only +, -, *, / and parentheses
    3. Evaluate to 24

    Args:
        expression: A string like '6 / (1 - 3 / 4)'
        numbers: List of four integers, e.g. [1, 3, 4, 6]

    Returns:
        dict with keys:
            'valid': bool -- whether the solution is correct
            'value': float or None -- what the expression evaluates to
            'error': str or None -- description of any error
    """
    # Clean the expression
    expr = expression.strip()

    # Remove any trailing " = 24" or similar
    expr = re.sub(r'\s*=\s*24\s*$', '', expr)

    # Check for disallowed characters (only digits, operators, parens, spaces, dots)
    if re.search(r'[^0-9+\-*/().\s]', expr):
        return {"valid": False, "value": None, "error": "Expression contains invalid characters"}

    # Extract all numbers from the expression
    expr_numbers = [int(x) for x in re.findall(r'\d+', expr)]

    # Check that the expression uses exactly the given numbers
    if sorted(expr_numbers) != sorted(numbers):
        return {
            "valid": False, "value": None,
            "error": f"Numbers don't match: expression uses {sorted(expr_numbers)}, expected {sorted(numbers)}"
        }

    # Evaluate the expression
    try:
        value = eval(expr)
    except Exception as e:
        return {"valid": False, "value": None, "error": f"Evaluation error: {e}"}

    # Check if it equals 24 (with floating point tolerance)
    if abs(value - 24) < 1e-6:
        return {"valid": True, "value": value, "error": None}
    else:
        return {"valid": False, "value": value, "error": f"Expression equals {value}, not 24"}


# Quick test on our known solutions
print("Verifying known solutions:")
for p in PUZZLES:
    result = verify_solution(p["solution"], p["numbers"])
    status = "PASS" if result["valid"] else f"FAIL ({result['error']})"
    print(f"  {str(p['numbers']):20s} {p['solution']:25s} -> {status}")

In [ ]:
# =============================================================================
# PROVIDED: Extract expression from model output
# =============================================================================

def extract_expression(response_text):
    """Try to extract an arithmetic expression from model output.

    Looks for common patterns:
    - 'Answer: <expr>'
    - '<expr> = 24'
    - Last line containing arithmetic operators

    Returns the extracted expression string, or the full response if no
    pattern matches.
    """
    text = response_text.strip()

    # Pattern 1: "Answer: <expr>"
    match = re.search(r'[Aa]nswer\s*[:=]\s*(.+)', text)
    if match:
        return match.group(1).strip()

    # Pattern 2: "<expr> = 24"
    match = re.search(r'([\d+\-*/()\s]+)\s*=\s*24', text)
    if match:
        return match.group(1).strip()

    # Pattern 3: last line with arithmetic operators
    for line in reversed(text.split('\n')):
        line = line.strip()
        if re.search(r'\d.*[+\-*/].*\d', line):
            # Clean common prefixes
            line = re.sub(r'^(So|Thus|Therefore|Hence|Final|Result)\s*[:,-]?\s*', '', line, flags=re.IGNORECASE)
            # Remove trailing "= 24" etc.
            line = re.sub(r'\s*=\s*\d+\.?\d*\s*$', '', line)
            return line.strip()

    return text


print("extract_expression() helper defined.")

---

## Section 1: Baseline: Single-Shot Reasoning

Our first approach is the simplest: give the model the puzzle in a single prompt and ask for a solution. No hints, no examples, no multi-step reasoning. Just one shot.

This establishes our **baseline** for comparison with more sophisticated methods.

**TODO 1.1:** Write a function `solve_single_shot(numbers)` that:
1. Constructs a prompt asking the model to solve the Game of 24 for the given numbers
2. Calls the model once (temperature=0)
3. Extracts the expression from the response
4. Returns a dict with the response, extracted expression, token usage, and verification result

Keep the prompt simple. Just describe the task and the numbers. Do **not** ask for step-by-step reasoning (that's Section 2).

In [ ]:
# TODO 1.1: Implement single-shot solving (~15-20 lines)
#
# Hint: Your prompt should clearly state:
#   - The rules of the Game of 24
#   - The four numbers to use
#   - That the answer should be an arithmetic expression
#
# Use call_model() with temperature=0.
# Use extract_expression() to parse the model's output.
# Use verify_solution() to check correctness.
#
# Return a dict with keys:
#   'raw_response', 'expression', 'result' (from verify_solution),
#   'total_tokens'

def solve_single_shot(numbers):
    """Solve Game of 24 with a single prompt."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Quick test
test = solve_single_shot([1, 2, 3, 4])
print(f"Expression: {test['expression']}")
print(f"Valid: {test['result']['valid']}")
print(f"Tokens: {test['total_tokens']}")

**TODO 1.2:** Run `solve_single_shot` on all puzzles and record the results.

In [ ]:
# TODO 1.2: Run single-shot on all puzzles (~10-15 lines)
#
# Loop over PUZZLES, call solve_single_shot() for each, and collect:
#   - Whether it solved the puzzle (result['valid'])
#   - Total tokens used
#
# Print a summary table showing:
#   Puzzle numbers | Difficulty | Correct? | Tokens | Model's expression
#
# At the end, print overall accuracy and total tokens used.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 2: Chain-of-Thought Prompting

Chain-of-thought (CoT) prompting asks the model to show its reasoning step by step before giving a final answer. The idea: by generating intermediate reasoning steps, the model can break a hard problem into easier sub-problems.

This is one of the simplest forms of inference-time compute: we're spending more tokens (and thus more compute) to get better answers.

**TODO 2.1:** Write a function `solve_chain_of_thought(numbers)` that:
1. Constructs a prompt that explicitly asks the model to think step by step
2. Calls the model once (temperature=0)
3. Extracts and verifies the final expression

Your prompt should encourage the model to:
- Consider which pairs of numbers to combine first
- Show each arithmetic step
- Give a final answer in a clear format like `Answer: <expression>`

In [ ]:
# TODO 2.1: Implement chain-of-thought solving (~20-25 lines)
#
# Key difference from single-shot: your prompt should ask the model to
# reason through the problem step by step before giving a final answer.
#
# You may want to increase max_tokens since CoT responses are longer.
#
# Return the same dict format as solve_single_shot.

def solve_chain_of_thought(numbers):
    """Solve Game of 24 with chain-of-thought prompting."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Quick test
test = solve_chain_of_thought([1, 5, 5, 5])
print(f"Response (first 300 chars): {test['raw_response'][:300]}")
print(f"\nExpression: {test['expression']}")
print(f"Valid: {test['result']['valid']}")
print(f"Tokens: {test['total_tokens']}")

**TODO 2.2:** Run `solve_chain_of_thought` on all puzzles and compare to the baseline.

In [ ]:
# TODO 2.2: Run CoT on all puzzles and print comparison (~15-20 lines)
#
# Same as TODO 1.2, but also compare against single-shot results.
# Print a table with columns:
#   Puzzle | Difficulty | Single-shot correct? | CoT correct? | CoT tokens
#
# At the end, print:
#   - Single-shot accuracy vs CoT accuracy
#   - Average tokens per puzzle for each method

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 3: Self-Consistency / Best-of-N

Self-consistency ([Wang et al., 2023](https://arxiv.org/abs/2203.11171)) is a simple but powerful idea: instead of taking the model's first answer, **sample multiple independent reasoning traces** and pick the most common answer.

The intuition: if the model arrives at the same answer via different reasoning paths, that answer is more likely to be correct.

For Game of 24, we have an even better option: since we can **verify** answers, we can use **best-of-N**: sample N solutions and return the first one that is verified correct.

**TODO 3.1:** Write a function `solve_best_of_n(numbers, n=5)` that:
1. Samples `n` reasoning traces using your CoT prompt (with temperature > 0)
2. Extracts and verifies each solution
3. Returns the **first valid solution** if one exists (best-of-N with a verifier)
4. If no valid solution is found, falls back to **majority vote** among the extracted expressions

This combines two ideas:
- **Best-of-N**: use the verifier to pick a correct answer
- **Self-consistency**: if verification fails for all N, fall back to majority vote

In [ ]:
# TODO 3.1: Implement best-of-N / self-consistency (~25-35 lines)
#
# Hint:
#   - Use call_model() with temperature=0.7 (or similar) to get diverse samples
#   - For each sample, extract the expression and run verify_solution()
#   - If any sample verifies as correct, return it immediately
#   - If none verify, collect all extracted expressions and do majority vote
#     (use collections.Counter to find the most common expression)
#   - Track total tokens across all N calls
#
# Return a dict with keys:
#   'expression', 'result', 'total_tokens', 'n_samples', 'n_valid',
#   'method' ('verified' if a valid solution was found, 'majority_vote' otherwise)

def solve_best_of_n(numbers, n=5):
    """Solve Game of 24 by sampling N solutions and picking the best."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Quick test
test = solve_best_of_n([3, 3, 8, 8], n=5)
print(f"Expression: {test['expression']}")
print(f"Valid: {test['result']['valid']}")
print(f"Method: {test['method']}")
print(f"Valid samples: {test['n_valid']}/{test['n_samples']}")
print(f"Total tokens: {test['total_tokens']}")

**TODO 3.2:** Run `solve_best_of_n` on all puzzles and compare to previous methods.

In [ ]:
# TODO 3.2: Run best-of-N on all puzzles (~15-20 lines)
#
# Print a comparison table:
#   Puzzle | Difficulty | Single-shot | CoT | Best-of-N | Tokens (Best-of-N)
#
# Print overall accuracy for all three methods.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 4: Tree-of-Thought Reasoning Agent

Now we build something more sophisticated: a **reasoning agent** that explores a tree of possible solutions.

The idea (from [Yao et al., 2023](https://arxiv.org/abs/2305.10601)):
1. **Propose**: Generate candidate next steps (e.g., "combine 3 and 8 to get 11")
2. **Evaluate**: Score each candidate. Does this partial solution look promising?
3. **Search**: Keep the best candidates and continue from there

This is a **control loop** around the model: instead of one pass, we run the model multiple times in a structured search.

### How it works for Game of 24

A **state** is a list of remaining numbers. We start with 4 numbers and combine two at a time:

```
State: [3, 3, 8, 8]
  -> Combine 3 and 8: [11, 3, 8], [5, 3, 8], [24, 3, 8], [3, 8, 0.375]
  -> Combine 3 and 3: [6, 8, 8], [0, 8, 8], [9, 8, 8], [1, 8, 8]
  -> Combine 8 and 8: [16, 3, 3], [0, 3, 3], [64, 3, 3], [1, 3, 3]
  -> ... (keep promising states, discard unpromising ones)
```

The key components:
- **State**: a list of remaining numbers
- **Propose**: enumerate which two numbers to combine and what operation to use
- **Evaluate**: ask the LLM to judge whether the remaining numbers can plausibly reach 24
- **Search**: beam search, keeping the top-k most promising states at each level

### 4.1 Propose: Generate candidate next steps

**TODO 4.1:** Write a function `propose_steps(numbers)` that generates all possible ways to combine two of the given numbers using +, -, \*, /.

In [ ]:
# TODO 4.1: Implement the propose step (~30-40 lines)
#
# Hint: You have two options:
#
# Option A (recommended): Generate candidates PROGRAMMATICALLY.
#   - For each pair of numbers (i, j), compute all four operations: i+j, i-j, i*j, i/j
#   - For each result, create a new state: [result] + remaining_numbers
#   - This is deterministic and exhaustive. No LLM needed for this step
#
# Option B: Ask the LLM to propose steps.
#   - Prompt: "Given numbers [a, b, c, d], list all ways to combine two numbers..."
#   - Parse the response to extract candidate states
#   - This is more interesting but less reliable
#
# We recommend Option A because it's more reliable and lets us focus the LLM's
# compute on the EVALUATION step (which is where reasoning really matters).
#
# Return: list of dicts, each with:
#   'operation': str describing the step (e.g., '3 + 8 = 11')
#   'new_numbers': list of remaining numbers (e.g., [11, 3, 8])

def propose_steps(numbers):
    """Generate all possible next steps by combining two numbers."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test
steps = propose_steps([3, 3, 8, 8])
print(f"Number of candidate steps: {len(steps)}")
for s in steps[:8]:
    print(f"  {s['operation']:20s} -> remaining: {s['new_numbers']}")

### 4.2 Evaluate: Score candidate states

**TODO 4.2:** Write a function `evaluate_state(numbers)` that asks the model whether the remaining numbers can plausibly be combined to reach 24.

In [ ]:
# TODO 4.2: Implement the evaluate step (~20-30 lines)
#
# Write a function that judges whether a set of numbers can reach 24.
#
# HINT: For small cases, you can check PROGRAMMATICALLY instead of asking the LLM:
#   - 1 number: just check if it equals 24
#   - 2 numbers: try all operations (a+b, a-b, b-a, a*b, a/b, b/a) and check
#   This is faster, cheaper, and perfectly accurate.
#
# For 3+ numbers, prompt the model with something like:
#   "Given the numbers [a, b, c], can you combine them using +, -, *, /
#    to reach 24? Answer with one word: 'sure', 'maybe', or 'impossible'."
#
# Parse the response to extract one of: 'sure', 'maybe', 'impossible'
# If the response doesn't match, default to 'maybe'.
#
# Return:
#   dict with 'score' (float: sure=1.0, maybe=0.5, impossible=0.0),
#   'judgment' (str), 'total_tokens' (int)

def evaluate_state(numbers):
    """Ask the LLM whether these numbers can reach 24."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test on a few states
for nums in [[24], [12, 2], [1, 2, 3], [100, 100, 100]]:
    result = evaluate_state(nums)
    print(f"  {str(nums):20s} -> {result['judgment']} (score={result['score']})")

### 4.3 Search: Putting it together

**TODO 4.3:** Write the main search function `solve_tree_of_thought(numbers, beam_width=3)`.

In [ ]:
# TODO 4.3: Implement tree-of-thought search (~40-55 lines)
#
# The search has 3 levels (since we start with 4 numbers):
#   Level 1: 4 numbers -> 3 numbers (combine two)
#   Level 2: 3 numbers -> 2 numbers (combine two)
#   Level 3: 2 numbers -> 1 number (combine the last two)
#
# At each level:
#   1. For each state in the current beam:
#      a. Generate all candidate next steps with propose_steps()
#      b. For states with 1 number: check if it equals 24
#      c. For states with 2+ numbers: evaluate with evaluate_state()
#   2. Sort all candidates by score
#   3. Keep the top beam_width candidates as the new beam
#
# Track the path of operations that led to each state so you can
# reconstruct the full solution.
#
# Return:
#   dict with 'solved' (bool), 'expression' (str or None),
#   'path' (list of operation strings), 'total_tokens' (int),
#   'states_explored' (int)

def solve_tree_of_thought(numbers, beam_width=3):
    """Solve Game of 24 using tree-of-thought search."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test on an easy puzzle
result = solve_tree_of_thought([1, 2, 3, 4])
print(f"Solved: {result['solved']}")
print(f"Path: {result['path']}")
print(f"States explored: {result['states_explored']}")
print(f"Total tokens: {result['total_tokens']}")

**TODO 4.4:** Run the tree-of-thought agent on all puzzles.

In [ ]:
# TODO 4.4: Run tree-of-thought on all puzzles (~15-20 lines)
#
# For each puzzle, run solve_tree_of_thought() and record:
#   - Whether it solved the puzzle
#   - The solution path
#   - Total tokens used
#   - Number of states explored
#
# Print a summary table and compare to all previous methods.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 5: Analysis and Reflection

Now let's compare all four methods and reflect on what we've learned.

### 5.1 Comparison Table

| Method | Accuracy | Avg Tokens/Puzzle | Total Tokens | Key Idea |
|--------|----------|-------------------|--------------|----------|
| Single-shot | _/10 | _ | _ | One prompt, one answer |
| Chain-of-thought | _/10 | _ | _ | Step-by-step reasoning |
| Best-of-N (N=5) | _/10 | _ | _ | Sample + verify / vote |
| Tree-of-Thought | _/10 | _ | _ | Propose + evaluate + search |

In [ ]:
# TODO 5.1: Create a summary visualization (~20-25 lines)
#
# Create a bar chart comparing:
#   - Accuracy (left y-axis)
#   - Average tokens per puzzle (right y-axis or separate subplot)
# for all four methods.
#
# This should clearly show the accuracy-vs-cost tradeoff.
#
# Hint: Use matplotlib with two subplots side by side:
#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

import matplotlib.pyplot as plt

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

### 5.2 Reflection Questions

Answer the following questions based on your experiments.

---

**Q1: When did extra test-time compute help?**

Look at which puzzles were solved by more expensive methods but not simpler ones. What made those puzzles harder? Is there a pattern relating puzzle difficulty to the benefit of additional compute?

*Your answer:*

*Write your answer here...*

**Q2: When did extra test-time compute hurt (or not help)?**

*Your answer:*

*Write your answer here...*

**Q3: What failure modes did you observe?**

*Your answer:*

*Write your answer here...*

**Q4: How is inference-time search different from training-time RLVR?**

*Your answer:*

*Write your answer here...*

**Q5: Scaling laws for inference compute**

*Your answer:*

*Write your answer here...*

---

## Optional Extension: Scaling Experiment

In [ ]:
# OPTIONAL: Scaling experiment (~20-30 lines)
#
# Pick 3-4 hard puzzles. For each value of N in [1, 3, 5, 10, 20]:
#   - Run solve_best_of_n(puzzle, n=N)
#   - Record accuracy and TOTAL tokens (not average per puzzle)
#
# Plot:
#   - Accuracy vs N
#   - Total tokens vs N
#
# Using total tokens (not avg) gives a cleaner picture:
# accuracy plateaus while cost keeps climbing — showing diminishing returns.
#
# This demonstrates the scaling law for inference-time compute.

# YOUR CODE HERE (optional)
pass

---

## Congratulations!

You have completed the reasoning agents lab. Here is a summary of what you built:

1. **Single-shot**: Baseline: one prompt, one answer. Fast but limited.
2. **Chain-of-thought**: Added step-by-step reasoning. Slightly better, slightly more tokens.
3. **Self-consistency / Best-of-N**: Sampled multiple solutions and used verification or voting. Significantly better, proportionally more tokens.
4. **Tree-of-Thought**: Built a reasoning agent with a propose-evaluate-search loop. Best accuracy, most tokens.

The big takeaway: **the same model can perform much better when wrapped in a smarter control loop.** This is the foundation of modern reasoning agents, and it's why inference-time compute is becoming as important as training-time compute in AI systems.

---

### Instructor Notes

**Key concepts reinforced:**
- Inference-time compute as a complement to training-time compute
- Chain-of-thought as the simplest form of test-time reasoning
- Self-consistency and the value of diverse sampling
- Tree-of-Thought and structured search over reasoning paths
- Agent control loops: propose, evaluate, search
- The accuracy-cost tradeoff in reasoning

**Expected results (approximate, will vary by model):**
- Single-shot: 3-5/10 correct
- CoT: 4-6/10 correct
- Best-of-5: 5-7/10 correct
- ToT: 6-9/10 correct

**Common student issues:**
- Expression parsing: model output varies in format. The extract_expression helper handles common cases but students may need to debug edge cases.
- API rate limits: if students hit rate limits, suggest reducing N or adding small delays.
- Token costs: total cost for the full lab should be well under $1 with an 8B model on OpenRouter.

**Discussion points for lecture tie-in:**
- How does this relate to AlphaGo's MCTS? (propose = policy network, evaluate = value network)
- How does this relate to o1/o3-style reasoning? (internal chain-of-thought, search over reasoning paths)
- What's the difference between search at inference time vs. RLVR at training time? (Both improve reasoning, but RLVR changes the model; search doesn't)